In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv("D:\Ai driven News credability and influence analysis\data\combined_news.csv")
df.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",1
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",0


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https\S+|www\S+","", text) # remove url
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove special chars
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

In [5]:
# apply clean text
df["clean_text"] = df["text"].apply(clean_text)
df[["text" , "clean_text"]].head()

,text,clean_text
0,"21st Century Wire says Ben Stein, reputable pr...",st century wire says ben stein reputable profe...
1,WASHINGTON (Reuters) - U.S. President Donald T...,washington reuters us president donald trump r...
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,reuters puerto rico governor ricardo rossello ...
3,"On Monday, Donald Trump once again embarrassed...",on monday donald trump once again embarrassed ...
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",glasgow scotland reuters most us presidential ...


In [6]:
# Text Length
df["text_length"] = df["clean_text"].apply(lambda x: len(x.split()))
df[["clean_text", "text_length"]].head()

,clean_text,text_length
0,st century wire says ben stein reputable profe...,170
1,washington reuters us president donald trump r...,767
2,reuters puerto rico governor ricardo rossello ...,303
3,on monday donald trump once again embarrassed ...,180
4,glasgow scotland reuters most us presidential ...,518


In [7]:
# Exclamation Count
df["exclamation_count"] = df["text"].apply(lambda x: str(x).count("!"))
df[["text", "exclamation_count"]].head()

,text,exclamation_count
0,"21st Century Wire says Ben Stein, reputable pr...",0
1,WASHINGTON (Reuters) - U.S. President Donald T...,0
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,0
3,"On Monday, Donald Trump once again embarrassed...",0
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",0


In [8]:
# Uppercase ratio
# str(text) > If text is NaN, number, or missing → prevents error.

def uppercase_ratio(text):
    text = str(text)
    if len(text) == 0:
        return 0
    upper_count = sum(1 for c in text if c.isupper())
    return upper_count / len(text)

df["uppercase_ratio"] = df["text"].apply(uppercase_ratio)
df[["text", "uppercase_ratio"]].head()

,text,uppercase_ratio
0,"21st Century Wire says Ben Stein, reputable pr...",0.101167
1,WASHINGTON (Reuters) - U.S. President Donald T...,0.045228
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,0.031385
3,"On Monday, Donald Trump once again embarrassed...",0.044212
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",0.039847


In [9]:
from textblob import TextBlob
# str(text) → converts text safely to string
# TextBlob(text) → creates a TextBlob object
# .sentiment.polarity → returns sentiment score

def sentiment_score(text):
    return TextBlob(str(text)).sentiment.polarity

df["sentiment_score"] = df["clean_text"].apply(sentiment_score)
df[["clean_text", "sentiment_score"]].head()

,clean_text,sentiment_score
0,st century wire says ben stein reputable profe...,0.096154
1,washington reuters us president donald trump r...,0.086597
2,reuters puerto rico governor ricardo rossello ...,-0.005044
3,on monday donald trump once again embarrassed ...,-0.011161
4,glasgow scotland reuters most us presidential ...,0.039347


In [15]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
# hstack → horizontally combines features

vectorizer = TfidfVectorizer(max_features=5000)
X_text = vectorizer.fit_transform(df["clean_text"])


numeric_features = df[
    ["text_length", "exclamation_count", "uppercase_ratio", "sentiment_score"]
].values

x = hstack([X_text, numeric_features])
y = df["label"]

In [16]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42, stratify = y)

In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter = 1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [18]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.98      0.99      0.99      4284
           1       0.99      0.99      0.99      4696

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

[[4259   25]
 [  69 4627]]


In [19]:
import pandas as pd

results = pd.DataFrame({
    "text" : df.loc[y_test.index, "text"],
    "clean_text" : df.loc[y_test.index, "clean_text"],
    "true_label" : y_test.values,
    "predicted_label" : y_pred
})

errors = results[results["true_label"] != results["predicted_label"]]
errors.head()

,text,clean_text,true_label,predicted_label
23000,WASHINGTON (Reuters) - First lady Michelle Oba...,washington reuters first lady michelle obama m...,0,1
44004,LOS ANGELES (Reuters) - The Republican preside...,los angeles reuters the republican presidentia...,0,1
2584,"Make no mistake about it, we are seeing tactic...",make no mistake about it we are seeing tactics...,1,0
44621,"A high profile, anti-mass migration Member of ...",a high profile antimass migration member of th...,1,0
21116,The largest armed U.S. military brigade to be ...,the largest armed us military brigade to be de...,1,0


In [20]:
# Credibility Score
probabilities = model.predict_proba(X_test)

# Probability of being Real (label = 1)
credibility_score = probabilities[:, 1]*100

In [21]:
cred_df = results.copy()
cred_df["credibility_score"] = credibility_score

cred_df[["text", "true_label", "predicted_label", "credibility_score"]].head()

,text,true_label,predicted_label,credibility_score
4426,Great parenting huh?,1,1,91.659737
10462,WASHINGTON (Reuters) - U.S. lawmakers introduc...,0,0,1.646889
36268,NEW YORK (Reuters) - President Donald Trump’s ...,0,0,2.836759
31665,HARARE (Reuters) - Retired army chief Constant...,0,0,2.250971
80,Donald Trump is the ultimate hypocrite and Fox...,1,1,99.858728
